## 01. IMPORTING LIBRARIES

In [6]:
from pathlib import Path
import json
import shutil
import pandas as pd

import joblib
import numpy as np
import sklearn

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)

## 02. PROJECT FOLDERS

In [7]:
# Works when the notebook is run from the project root or a folder beneath it.
PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "Data" / "Model_Ready").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find Data/Model_Ready. Open this notebook from the project folder."
    )

MODEL_READY_DIR = PROJECT_ROOT / "Data" / "Model_Ready"
FEATURE_DIR = MODEL_READY_DIR / "Features"
TARGET_DIR = MODEL_READY_DIR / "Targets"
PREPROCESSOR_PATH = MODEL_READY_DIR / "Metadata" / "preprocessor.joblib"

MODEL_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Models" / "Final"
RESULTS_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Results" / "Final_Evaluation"

print("Project root:", PROJECT_ROOT)
print("Feature folder:", FEATURE_DIR)
print("Target folder:", TARGET_DIR)
print("Preprocessor:", PREPROCESSOR_PATH)

Project root: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor
Feature folder: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Features
Target folder: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Targets
Preprocessor: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Metadata\preprocessor.joblib


## 03. CONFIRM REQUIRED FILES

In [8]:
required_files = {
    "X_train": FEATURE_DIR / "X_train.csv",
    "X_validation": FEATURE_DIR / "X_validation.csv",
    "X_test": FEATURE_DIR / "X_test.csv",
    "y_train_reg": TARGET_DIR / "y_train_regression.csv",
    "y_validation_reg": TARGET_DIR / "y_validation_regression.csv",
    "y_test_reg": TARGET_DIR / "y_test_regression.csv",
    "y_train_cls": TARGET_DIR / "y_train_classification.csv",
    "y_validation_cls": TARGET_DIR / "y_validation_classification.csv",
    "y_test_cls": TARGET_DIR / "y_test_classification.csv",
    "preprocessor": PREPROCESSOR_PATH,
}

file_check = pd.DataFrame(
    [
        {"file": name, "exists": path.is_file(), "path": str(path)}
        for name, path in required_files.items()
    ]
)

display(file_check)

missing_files = file_check.loc[~file_check["exists"], "path"].tolist()
if missing_files:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_files))

,file,exists,path
0,X_train,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
1,X_validation,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
2,X_test,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
3,y_train_reg,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
4,y_validation_reg,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
5,y_test_reg,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
6,y_train_cls,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
7,y_validation_cls,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
8,y_test_cls,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...
9,preprocessor,True,c:\Users\bongo\OneDrive\Desktop\GitHub\Movie A...


## 04. LOAD THE SAVED SPLITS

In [9]:
X_train = pd.read_csv(required_files["X_train"])
X_validation = pd.read_csv(required_files["X_validation"])
X_test = pd.read_csv(required_files["X_test"])

def load_target(path, column):
    frame = pd.read_csv(path)
    if column not in frame.columns:
        raise KeyError(f"{path.name} must contain '{column}'. Found: {frame.columns.tolist()}")
    return frame[column]

y_train_reg = load_target(required_files["y_train_reg"], "log_worldwide_box_office")
y_validation_reg = load_target(
    required_files["y_validation_reg"], "log_worldwide_box_office"
)
y_test_reg = load_target(required_files["y_test_reg"], "log_worldwide_box_office")

y_train_cls = load_target(required_files["y_train_cls"], "blockbuster")
y_validation_cls = load_target(required_files["y_validation_cls"], "blockbuster")
y_test_cls = load_target(required_files["y_test_cls"], "blockbuster")

split_sizes = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "feature_rows": [len(X_train), len(X_validation), len(X_test)],
        "regression_target_rows": [
            len(y_train_reg), len(y_validation_reg), len(y_test_reg)
        ],
        "classification_target_rows": [
            len(y_train_cls), len(y_validation_cls), len(y_test_cls)
        ],
    }
)

display(split_sizes)

,split,feature_rows,regression_target_rows,classification_target_rows
0,Train,6668,6668,6668
1,Validation,600,600,600
2,Test,211,211,211


## 05. CHECKING SPLIT ALIGNMENT AND FEATURE SCHEMA

In [10]:
for split_name, X, y_reg, y_cls in [
    ("train", X_train, y_train_reg, y_train_cls),
    ("validation", X_validation, y_validation_reg, y_validation_cls),
    ("test", X_test, y_test_reg, y_test_cls),
]:
    if not (len(X) == len(y_reg) == len(y_cls)):
        raise ValueError(f"{split_name} features and targets have different row counts.")

if X_train.columns.tolist() != X_validation.columns.tolist():
    raise ValueError("Training and validation feature columns differ.")

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError("Training and test feature columns differ.")

for split_name, X, y_reg, y_cls in [
    ("train", X_train, y_train_reg, y_train_cls),
    ("validation", X_validation, y_validation_reg, y_validation_cls),
    ("test", X_test, y_test_reg, y_test_cls),
]:
    if X.columns.duplicated().any():
        raise ValueError(f"{split_name} contains duplicate feature columns.")
    if not np.isfinite(y_reg.to_numpy(dtype=float)).all():
        raise ValueError(f"{split_name} contains invalid regression targets.")
    if not set(y_cls.dropna().unique()).issubset({0, 1}):
        raise ValueError(f"{split_name} has classification labels outside 0 and 1.")

print("Split alignment and feature schema: PASS")
print("Number of processed features:", X_train.shape[1])

Split alignment and feature schema: PASS
Number of processed features: 36


## 06. REBUILDING THE LOCKED MODELS

In [11]:
best_threshold = 0.57

best_regression_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

best_classification_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

best_regression_model.fit(X_train, y_train_reg)
best_classification_model.fit(X_train, y_train_cls)

print("Locked regression model fitted.")
print("Locked classification model fitted.")
print(f"Locked classification threshold: {best_threshold:.2f}")

c:\Users\bongo\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\bongo\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\bongo\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bongo\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Locked regression model fitted.
Locked classification model fitted.
Locked classification threshold: 0.57


## #Validation

In [12]:
validation_reg_pred = best_regression_model.predict(X_validation)

positive_class_index = list(best_classification_model.classes_).index(1)
validation_probability = best_classification_model.predict_proba(X_validation)[
    :, positive_class_index
]
validation_cls_pred = (validation_probability >= best_threshold).astype(int)

validation_check = pd.Series(
    {
        "regression_rmse": np.sqrt(
            mean_squared_error(y_validation_reg, validation_reg_pred)
        ),
        "classification_roc_auc": roc_auc_score(
            y_validation_cls, validation_probability
        ),
        "classification_f1_at_0.57": f1_score(
            y_validation_cls, validation_cls_pred
        ),
    },
    name="reproduced",
)

validation_expected = pd.Series(
    {
        "regression_rmse": 1.414490,
        "classification_roc_auc": 0.915272,
        "classification_f1_at_0.57": 0.735178,
    },
    name="notebook_08",
)

validation_comparison = pd.concat(
    [validation_expected, validation_check], axis=1
)
validation_comparison["absolute_difference"] = (
    validation_comparison["reproduced"]
    - validation_comparison["notebook_08"]
).abs()

display(validation_comparison)

if (validation_comparison["absolute_difference"] > 0.001).any():
    raise ValueError(
        "Validation results do not reproduce Notebook 08. "
        "Check the saved splits and model settings before evaluating the test set."
    )

print("Validation reproduction: PASS")

,notebook_08,reproduced,absolute_difference
regression_rmse,1.414490,1.414490,3.737379e-07
classification_roc_auc,0.915272,0.915272,1.884165e-07
classification_f1_at_0.57,0.735178,0.735178,1.343874e-07


Validation reproduction: PASS


## 07. FINAL REGRESSION EVALUATION ON THE UNTOUCHED TEST SET

In [13]:
test_reg_pred_log = best_regression_model.predict(X_test)

y_test_reg_array = y_test_reg.to_numpy(dtype=float)

# The target was created using np.log1p(worldwide_box_office).
test_revenue_actual = np.expm1(y_test_reg_array)
test_revenue_pred = np.clip(np.expm1(test_reg_pred_log), 0, None)

regression_test_metrics = {
    "log_mae": mean_absolute_error(y_test_reg_array, test_reg_pred_log),
    "log_rmse": np.sqrt(
        mean_squared_error(y_test_reg_array, test_reg_pred_log)
    ),
    "log_r2": r2_score(y_test_reg_array, test_reg_pred_log),
    "revenue_mae_usd": mean_absolute_error(
        test_revenue_actual, test_revenue_pred
    ),
    "revenue_rmse_usd": np.sqrt(
        mean_squared_error(test_revenue_actual, test_revenue_pred)
    ),
}

display(
    pd.Series(regression_test_metrics, name="test_value").to_frame()
)

,test_value
log_mae,1.437716e+00
log_rmse,2.005540e+00
log_r2,5.426756e-01
revenue_mae_usd,9.323080e+07
revenue_rmse_usd,2.168016e+08


## 08. FINAL CLASSIFICATION EVALUATION ON THE UNTOUCHED TEST SET

In [14]:
test_blockbuster_probability = best_classification_model.predict_proba(
    X_test
)[:, positive_class_index]

test_blockbuster_pred = (
    test_blockbuster_probability >= best_threshold
).astype(int)

classification_test_metrics = {
    "threshold": best_threshold,
    "accuracy": accuracy_score(y_test_cls, test_blockbuster_pred),
    "precision": precision_score(
        y_test_cls, test_blockbuster_pred, zero_division=0
    ),
    "recall": recall_score(
        y_test_cls, test_blockbuster_pred, zero_division=0
    ),
    "f1": f1_score(
        y_test_cls, test_blockbuster_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test_cls, test_blockbuster_probability
    ),
}

display(
    pd.Series(classification_test_metrics, name="test_value").to_frame()
)

test_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test_cls, test_blockbuster_pred, labels=[0, 1]),
    index=["Actual non-blockbuster", "Actual blockbuster"],
    columns=["Predicted non-blockbuster", "Predicted blockbuster"],
)

display(test_confusion_matrix)

print(
    classification_report(
        y_test_cls,
        test_blockbuster_pred,
        target_names=["Non-blockbuster", "Blockbuster"],
        zero_division=0,
    )
)

,test_value
threshold,0.570000
accuracy,0.881517
precision,0.804878
recall,0.660000
f1,0.725275
roc_auc,0.904348


,Predicted non-blockbuster,Predicted blockbuster
Actual non-blockbuster,153,8
Actual blockbuster,17,33


                 precision    recall  f1-score   support

Non-blockbuster       0.90      0.95      0.92       161
    Blockbuster       0.80      0.66      0.73        50

       accuracy                           0.88       211
      macro avg       0.85      0.81      0.82       211
   weighted avg       0.88      0.88      0.88       211



## 09. ASSEMBLE ROW-LEVEL TEST RESULTS

In [15]:
test_predictions = pd.DataFrame(
    {
        "test_row": np.arange(len(X_test)),
        "actual_log_worldwide_box_office": y_test_reg_array,
        "predicted_log_worldwide_box_office": test_reg_pred_log,
        "actual_worldwide_box_office_usd": test_revenue_actual,
        "predicted_worldwide_box_office_usd": test_revenue_pred,
        "actual_blockbuster": y_test_cls.to_numpy(dtype=int),
        "blockbuster_probability": test_blockbuster_probability,
        "predicted_blockbuster": test_blockbuster_pred,
        "classification_threshold": best_threshold,
    }
)

display(test_predictions.head())

,test_row,actual_log_worldwide_box_office,predicted_log_worldwide_box_office,actual_worldwide_box_office_usd,predicted_worldwide_box_office_usd,actual_blockbuster,blockbuster_probability,predicted_blockbuster,classification_threshold
0,0,17.686037,16.684469,4.796764e+07,1.761865e+07,0,0.068483,0,0.57
1,1,19.741278,19.571765,3.745658e+08,3.161618e+08,1,0.862064,1,0.57
2,2,21.751992,20.646052,2.797501e+09,9.256921e+08,1,0.893682,1,0.57
3,3,17.524485,16.910057,4.081192e+07,2.207721e+07,0,0.159642,0,0.57
4,4,19.193569,18.206281,2.166012e+08,8.070257e+07,1,0.445755,0,0.57


## 1O. SAVING

In [16]:
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

regression_model_path = MODEL_OUTPUT_DIR / "revenue_regressor.joblib"
classification_model_path = MODEL_OUTPUT_DIR / "blockbuster_classifier.joblib"
saved_preprocessor_path = MODEL_OUTPUT_DIR / "preprocessor.joblib"

regression_metrics_path = RESULTS_OUTPUT_DIR / "regression_test_metrics.json"
classification_metrics_path = RESULTS_OUTPUT_DIR / "classification_test_metrics.json"
predictions_path = RESULTS_OUTPUT_DIR / "test_predictions.csv"
confusion_matrix_path = RESULTS_OUTPUT_DIR / "classification_confusion_matrix.csv"
manifest_path = MODEL_OUTPUT_DIR / "model_manifest.json"

joblib.dump(best_regression_model, regression_model_path)
joblib.dump(best_classification_model, classification_model_path)
shutil.copy2(PREPROCESSOR_PATH, saved_preprocessor_path)

with regression_metrics_path.open("w", encoding="utf-8") as file:
    json.dump(
        {key: float(value) for key, value in regression_test_metrics.items()},
        file,
        indent=2,
    )

with classification_metrics_path.open("w", encoding="utf-8") as file:
    json.dump(
        {key: float(value) for key, value in classification_test_metrics.items()},
        file,
        indent=2,
    )

test_predictions.to_csv(predictions_path, index=False)
test_confusion_matrix.to_csv(confusion_matrix_path)

manifest = {
    "regression_model": regression_model_path.name,
    "classification_model": classification_model_path.name,
    "preprocessor": saved_preprocessor_path.name,
    "classification_threshold": best_threshold,
    "regression_target": "log_worldwide_box_office",
    "classification_target": "blockbuster",
    "regression_inverse_transform": "np.expm1",
    "model_training_split": "train",
    "model_selection_split": "validation",
    "final_evaluation_split": "test",
    "processed_feature_columns": X_train.columns.tolist(),
    "scikit_learn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Saved models to:", MODEL_OUTPUT_DIR)
print("Saved final evaluation results to:", RESULTS_OUTPUT_DIR)

Saved models to: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Models\Final
Saved final evaluation results to: c:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Results\Final_Evaluation


## CHECKING SAVED MODELS REPRODUCED PREDICTIONS

In [17]:
loaded_regressor = joblib.load(regression_model_path)
loaded_classifier = joblib.load(classification_model_path)

sample = X_test.head(min(5, len(X_test)))

np.testing.assert_allclose(
    loaded_regressor.predict(sample),
    best_regression_model.predict(sample),
)

np.testing.assert_allclose(
    loaded_classifier.predict_proba(sample),
    best_classification_model.predict_proba(sample),
)

print("Saved-model prediction check: PASS")

Saved-model prediction check: PASS
